In [0]:
import json
import mlflow
import pyspark.sql.functions as sf

from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.state import CompiledStateGraph

mlflow.langchain.autolog()

from databricks.vector_search.client import VectorSearchClient
from agent import build_agent, _load_config

In [0]:
df_bronze_products_info = (
  spark
  .read
  .table("workspace.sephora_products_and_skincare_reviews.bronze_products_info")
)
display(df_bronze_products_info.limit(5))

In [0]:
df_bronze_reviews = (
  spark
  .read
  .table("workspace.sephora_products_and_skincare_reviews.bronze_reviews")
)
display(df_bronze_reviews.limit(5))

In [0]:
df_bronze_reviews_selected = df_bronze_reviews.select(
  "product_id", "rating", "is_recommended", "review_text", "price_usd"
)

df_bronze_products_recommendations = (
  df_bronze_products_info
  .select("product_id", "product_name", "primary_category", "secondary_category", "tertiary_category", "highlights", )
  .join(
    df_bronze_reviews_selected,
    on="product_id",
    how="inner"
  )
  .withColumn("id", sf.monotonically_increasing_id())
)
display(df_bronze_products_recommendations.limit(5))

In [0]:
(
    df_bronze_products_recommendations
    .write
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("workspace.sephora_products_and_skincare_reviews.bronze_products_recommendations")
)

In [0]:
%sql
USE workspace.sephora_products_and_skincare_reviews;

ALTER TABLE bronze_products_recommendations
  SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

SELECT * FROM bronze_products_recommendations;

In [0]:
%sql
USE workspace.sephora_products_and_skincare_reviews;

SELECT
  product_id,
  count(rating),
  avg(try_cast(rating AS NUMERIC)),
  min(rating),
  max(rating),
  abs(2.5-avg(try_cast(rating AS NUMERIC))) as diff
FROM bronze_products_recommendations
GROUP BY product_id
ORDER BY diff ASC;

In [0]:
product_id = (
    df_bronze_products_recommendations
    .withColumn("rating", sf.col("rating").try_cast("double"))
    .groupBy(sf.col("product_id"))
    .agg(
        sf.count(sf.col("rating")).alias("cnt"),
        sf.avg(sf.col("rating")).alias("rating_avg"),
        sf.min(sf.col("rating")).alias("rating_min"),
        sf.max(sf.col("rating")).alias("rating_max"),
    )
    .filter(
        (sf.col("rating_min") == 1.0)
        & (sf.col("rating_max") == 5.0)
    )
    .withColumn(
        "diff",
        sf.abs(
            sf.lit(2.5)
            - sf.col("rating_avg")
        )
    )
    .orderBy(sf.col("diff"))
    .filter(sf.col("cnt").between(5, 10))
    .select("product_id")
).collect()[0][0]

display(product_id)

In [0]:
df_silver_products_recommendations = (
    df_bronze_products_recommendations
    .where(sf.col("product_id") == product_id)
)

display(df_silver_products_recommendations)

In [0]:
vsc = VectorSearchClient()

vector_search_endpoint = "sephora"
index_name = "workspace.sephora_products_and_skincare_reviews.silver_products_recommendations_index"
table_name = "workspace.sephora_products_and_skincare_reviews.silver_products_recommendations"

In [0]:
(
    df_silver_products_recommendations
    .write
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(table_name)
)

In [0]:
deploy_client = mlflow.deployments.get_deploy_client("databricks")

question = "How buyers describe this product in their review?"
response = deploy_client.predict(
    endpoint="databricks-gte-large-en",
    inputs={"input": [question]}
)

embeddings = [
    e["embedding"]
    for e
    in  response.data
]

In [0]:
print("Embedding shape:", len(embeddings[0]))
print("Embeddings for question:", embeddings[0])

In [0]:
vsc.create_delta_sync_index_and_wait(
    endpoint_name=vector_search_endpoint,
    index_name=index_name,
    source_table_name=table_name,
    primary_key="id",
    embedding_source_column="review_text",
    embedding_model_endpoint_name="databricks-gte-large-en",
    pipeline_type="TRIGGERED",
)

In [0]:
index = vsc.get_index(index_name=index_name)
print(
    json.dumps(
        index.describe(), 
        indent=4
))

In [0]:
query_text = "Reviewer has mixed feelings about the product"
results = index.similarity_search(
    query_text=query_text,
    columns=["product_id", "product_name", "review_text"],
    num_results=3,
)

print(json.dumps(results, indent=4))

In [0]:
query_text = "Reviewer has VERY negative feelings about the product"
results = index.similarity_search(
    query_text=query_text,
    columns=["product_id", "product_name", "review_text"],
    num_results=3,
)

print(json.dumps(results, indent=4))

In [0]:
query_text = "Reviewer has VERY POSITIVE feelings about the product"
results = index.similarity_search(
    query_text=query_text,
    columns=["product_id", "product_name", "review_text"],
    num_results=3,
)

print(json.dumps(results, indent=4))

In [0]:
query_text = "Reviewer has VERY POSITIVE feelings about the product"
results = index.similarity_search(
    query_text=query_text,
    query_type="hybrid",
    columns=["product_id", "product_name", "review_text"],
    num_results=3,
)

print(json.dumps(results, indent=4))

In [0]:
query_text = "Awkward, useless"
results = index.similarity_search(
    query_text=query_text,
    query_type="FULL_TEXT",
    columns=["product_id", "product_name", "review_text"],
    num_results=3,
)

print(json.dumps(results, indent=4))

In [0]:
config = _load_config("agent_config_reviews.yaml")

agent = build_agent(
    llm_endpoint=config["llm_endpoint_name"],
    index_name=config["vs_index_name"],
    num_results=config["vs_num_results"],
    description=config["description"],
    system_prompt=config["system_prompt"]
)

In [0]:
# Rebuild agent fresh to ensure we have the latest configuration
config = {"configurable": {"thread_id": "review_agent"}}
response = agent.invoke(
    input={
        "messages": [
            {
                "role": "user",
                "content": "What are top 3 MOST POSITIVE reviews?"
            },
            {
                "role": "user",
                "content": "What are top 3 MOST NEGATIVE reviews?"
            }
        ]
    },
    config=config
)
print(response["messages"][-1].content)

**Top 3 Most Positive Reviews (from the available data)**  

| Rank | Review excerpt | Key positive points | Source |
|------|----------------|---------------------|--------|
| 1 | “I use this with my Shiseido eye cream and eye serums and the next morning my eyes are so rested and plumped! **Love it**.” | Clear enthusiasm (“Love it”), noticeable improvement in eye appearance, repeat‑use with other products. | Document id 648370 |
| 2 | “Using the **GloPro** on my under‑eye skin has made a **big difference** for me. I don’t necessarily think the eye attachment is 100 % necessary … might just be a practice thing on my end.” | Strong impact statement (“big difference”), overall satisfaction despite a minor usability comment. | Document id 648367 |
| 3 | “Too intense for the sensitive skin around my eyes but **works great on my face**.” | Positive outcome for facial use; the only negative note relates to eye‑area intensity, not the overall product effectiveness. | Document id 648366 |

**Top 3 Most Negative Reviews (from the available data)**  

The current dataset contains only a single review that includes a negative sentiment (Document id 648366). Since